# E043 — Image TTA (Flip + Small Rotations)

**Motivation:** E030 (rotation TTA) hurt performance, but that was with large rotations (+/-20deg). Small rotations (+/-5deg) + flip might help with E033's adversarial training.

**Hypothesis:** TTA with flip + small rotations (+/-5deg) will improve over E033 single-view scoring.

**Configs:**
- `single`: E033 baseline (no TTA)
- `flip`: Horizontal flip only
- `flip+rot5`: Flip + rotations -5,0,+5 deg (5 views)

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
from PIL import Image
from scipy.ndimage import rotate
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
N_PCA = 50

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E033_REF = {'mean_eer': 0.51, 'std_eer': 0.36}

In [ ]:
def _find_png(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".png")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _load_image(path):
    img = Image.open(path).convert("RGB")
    if img.size != (80, 80):
        img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2)

def train_e033_model(train_df, data_dir, seed):
    from scipy.ndimage import rotate as rotate_fn
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _, row in train_df.iterrows():
        orig = _load_image(_find_png(row["stem"], data_dir))
        vecs = [orig, orig[:, ::-1].copy()]
        vecs += [np.clip(orig * rng.uniform(0.7, 1.3), 0, 255)]
        vecs += [np.clip(orig + rng.normal(0, 15, orig.shape), 0, 255)]
        if row["label"] == 1:
            for angle in [-10, -5, 5, 10]:
                vecs.append(rotate_fn(orig, angle, reshape=False, order=1, mode='constant', cval=0))
        for v in vecs:
            X.append(v.flatten()); y.append(row["label"])
    X, y = np.array(X), np.array(y)
    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    clf.fit(pca.fit_transform(scaler.fit_transform(X)), y)
    return scaler, pca, clf

def score_tta(png_path, scaler, pca, clf, tta_type):
    x = _load_image(png_path)
    views = [x]
    if tta_type in ['flip', 'flip+rot5']:
        views.append(x[:, ::-1].copy())
    if tta_type == 'flip+rot5':
        for angle in [-5, 5]:
            views.append(rotate(x, angle, reshape=False, order=1, mode='constant', cval=0))
    scores = []
    for v in views:
        v_flat = v.flatten().reshape(1, -1)
        v_pca = pca.transform(scaler.transform(v_flat))
        scores.append(float(clf.decision_function(v_pca)[0]))
    return np.mean(scores)

print('Functions defined')

## 2. Cross-validation

In [ ]:
tta_types = ['single', 'flip', 'flip+rot5']
results = {}

for tta in tta_types:
    print(f"\n=== {tta} ===")
    fold_eers = []
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        seed_f = SEED + fold_id
        train_df = manifest.loc[train_idx]
        val_df = manifest.loc[val_idx]
        scaler, pca, clf = train_e033_model(train_df, DATA, seed_f)
        scores = [score_tta(_find_png(row["stem"], DATA), scaler, pca, clf, tta) for _, row in val_df.iterrows()]
        scores = np.array(scores)
        eer, _ = compute_eer(scores[y_all[val_idx] == 1], scores[y_all[val_idx] == 0])
        fold_eers.append(eer * 100)
        print(f"  Fold {fold_id}: EER={eer*100:.2f}%")
    results[tta] = {'fold_eers': fold_eers, 'mean': np.mean(fold_eers), 'std': np.std(fold_eers)}
    print(f"  Mean +/- Std: {np.mean(fold_eers):.2f} +/- {np.std(fold_eers):.2f}%")

print("\n=== Summary ===")
for tta, res in results.items():
    imp = results['single']['mean'] - res['mean']
    print(f"{tta:15s}: {res['mean']:5.2f} +/- {res['std']:5.2f}%  (improvement: {imp:+.2f}pp)")

## 3. Conclusion

Image TTA: [TBD]
Decision: [Adopt/Reject]